# Restaurant Resource Planning System

## Step 1: Problem Breakdown

We need to build a system that predicts three operational outputs for any upcoming day.

1. Hourly covers: how many customers are expected in each hour.
2. Staffing needs: how many people should be scheduled by role and station.
3. Ingredient ordering: how much of each ingredient should be ordered after accounting for shelf life and supplier lead times.

This problem includes both machine learning and business logic.

Machine learning is used to forecast hourly customer demand. That forecast drives the rest of the planning system. Staffing and ingredient ordering are derived from the demand forecast using operational rules, configuration, and constraints from the business.

The critical requirement is that this is not a one-time prediction system. Forecasts will be wrong in real operations. Managers must be able to record corrections such as lower-than-expected traffic because of rain or higher-than-expected demand because of a local event. The system should store predicted values and actual outcomes, measure forecast error over time, and use that feedback to improve future predictions.

For the first version, we assume a single restaurant location, hourly forecasting granularity, fixed restaurant operating hours, role-based staffing rules, and ingredient planning based on menu-to-ingredient mappings. The overall goal is to build a practical feedback loop that becomes more accurate as more operational data is collected.

## Step 2: Data Design

The system needs a compact data model that connects demand forecasting, staffing, ingredient planning, and feedback.

This notebook uses six core datasets:

- `historical_sales.csv` for hourly demand, sales, reservations, walk-ins, and promotions.
- `external_features.csv` for weather, holidays, events, and weekend signals.
- `staff_roles.csv` for staffing rules by role and station.
- `menu_ingredients.csv` for mapping dishes to ingredient consumption.
- `ingredient_master.csv` for shelf life, lead time, and minimum order constraints.
- `feedback.csv` for predicted versus actual covers and manager corrections.

These files are enough for the first version of the system because they let us forecast covers, convert demand into staffing needs, estimate ingredient requirements, and then improve the model through feedback.

Sample record structure:

```json
{
  "historical_sales": {
    "timestamp": "2026-01-01 12:00:00",
    "covers": 25,
    "reservations": 10,
    "walk_ins": 15,
    "promotion_flag": 0
  },
  "external_features": {
    "timestamp": "2026-01-01 12:00:00",
    "holiday_flag": 1,
    "event_flag": 0,
    "temp_c": 31.3,
    "rain_mm": 0.0
  },
  "feedback": {
    "timestamp": "2026-06-23 12:00:00",
    "predicted_covers": 12,
    "actual_covers": 12,
    "manager_note": "Good turnout"
  }
}
```

In [ ]:
# Data path
from pathlib import Path
DATA_DIR = Path('V1Data')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

np.random.seed(42)
DAYS_OF_DATA = 180
START_DATE = datetime(2026, 1, 1)

timestamps = [START_DATE + timedelta(hours=i) for i in range(DAYS_OF_DATA * 24)]
timestamps = [t for t in timestamps if 11 <= t.hour <= 22]

def gen_external(ts_list):
    data = []
    for t in ts_list:
        is_weekend = 1 if t.weekday() >= 4 else 0
        temp = np.random.normal(25, 5) + (5 if 12 <= t.hour <= 16 else 0)
        rain = np.random.choice([0, 0, 0, 5, 15], p=[0.7, 0.15, 0.1, 0.04, 0.01])
        holiday_flag = 1 if (t.month == 12 and t.day == 25) or (t.month == 1 and t.day == 1) else 0
        event_flag = 1 if np.random.random() < 0.05 else 0

        data.append({
            'timestamp': t,
            'holiday_flag': holiday_flag,
            'event_flag': event_flag,
            'holiday_name': 'Public Holiday' if holiday_flag else '',
            'event_name': 'Local Event' if event_flag else '',
            'temp_c': round(temp, 1),
            'rain_mm': float(rain),
            'is_weekend': is_weekend
        })
    return pd.DataFrame(data)

def gen_sales(df_ext):
    data = []
    for _, row in df_ext.iterrows():
        base = 20 if 12 <= row['timestamp'].hour <= 14 or 19 <= row['timestamp'].hour <= 21 else 10
        boost = (row['is_weekend'] * 15) + (row['event_flag'] * 20) - (row['rain_mm'] * 0.5)
        promo = 1 if np.random.random() < 0.1 else 0

        covers = int(max(5, base + boost + (promo * 10) + np.random.normal(0, 5)))
        res = int(covers * np.random.uniform(0.3, 0.6))
        walk_ins = covers - res
        avg_ticket = round(np.random.normal(22, 3), 2)

        data.append({
            'timestamp': row['timestamp'],
            'covers': covers,
            'gross_sales': round(covers * avg_ticket, 2),
            'num_orders': int(covers * 0.9),
            'avg_ticket_size': avg_ticket,
            'reservations': res,
            'walk_ins': walk_ins,
            'promotion_flag': promo
        })
    return pd.DataFrame(data)

def gen_feedback(df_s):
    recent = df_s.tail(84).copy()
    recent['predicted_covers'] = (recent['covers'] + np.random.normal(0, 3, len(recent))).astype(int)
    recent['manager_note'] = np.where(recent['covers'] < recent['predicted_covers'], 'Lower than expected', 'Good turnout')
    return recent[['timestamp', 'predicted_covers', 'covers', 'manager_note']].rename(columns={'covers': 'actual_covers'})

df_external = gen_external(timestamps)
df_sales = gen_sales(df_external)

staff_data = [
    ['server', 'dining', 18, 1, 8, 6, 15.0],
    ['cook', 'grill', 25, 1, 6, 6, 18.0],
    ['cook', 'fryer', 20, 1, 4, 6, 17.0],
    ['manager', 'floor', 40, 1, 2, 8, 25.0]
]
menu_data = [
    ['M001', 'Chicken Burger', 'Main', 'I001', 'Chicken Patty', 1, 'piece', 0.05],
    ['M001', 'Chicken Burger', 'Main', 'I002', 'Burger Bun', 1, 'piece', 0.02],
    ['M002', 'French Fries', 'Side', 'I003', 'Potato', 0.2, 'kg', 0.10],
    ['M002', 'French Fries', 'Side', 'I004', 'Oil', 0.05, 'litre', 0.08]
]
ingredient_master = [
    ['I001', 'Chicken Patty', 2, 1, 50, 'piece', 'freezer'],
    ['I002', 'Burger Bun', 1, 1, 100, 'piece', 'ambient'],
    ['I003', 'Potato', 5, 2, 20, 'kg', 'ambient'],
    ['I004', 'Oil', 30, 3, 10, 'litre', 'ambient']
]

df_staff = pd.DataFrame(staff_data, columns=['role', 'station', 'covers_per_hour_per_staff', 'min_staff_per_shift', 'max_staff_per_shift', 'shift_length_hours', 'hourly_cost'])
df_menu = pd.DataFrame(menu_data, columns=['menu_item_id', 'menu_item_name', 'category', 'ingredient_id', 'ingredient_name', 'qty_per_dish', 'uom', 'yield_loss_pct'])
df_ing_master = pd.DataFrame(ingredient_master, columns=['ingredient_id', 'ingredient_name', 'shelf_life_days', 'lead_time_days', 'min_order_qty', 'uom', 'storage_type'])
df_feedback = gen_feedback(df_sales)

df_external.to_csv(DATA_DIR / 'external_features.csv', index=False)
df_sales.to_csv(DATA_DIR / 'historical_sales.csv', index=False)
df_staff.to_csv(DATA_DIR / 'staff_roles.csv', index=False)
df_menu.to_csv(DATA_DIR / 'menu_ingredients.csv', index=False)
df_ing_master.to_csv(DATA_DIR / 'ingredient_master.csv', index=False)
df_feedback.to_csv(DATA_DIR / 'feedback.csv', index=False)

print(f'Successfully generated all 6 CSV files in {DATA_DIR.resolve()}.')
df_sales.head()


## Step 3: Feature Engineering

Feature engineering converts raw hourly records into signals that the forecasting model can learn from. For restaurant demand, the most useful features are usually time patterns, recent demand history, short-term trends, and external context such as weather or local events.

In this step, we merge sales data with external features and create four groups of predictors:

- Time-based features such as hour, day of week, month, weekend flag, and cyclical hour encoding.
- Lag features that capture previous demand at the last hour, previous day, and previous week.
- Rolling features that summarize recent demand trends and volatility.
- External and business signals such as weather, holiday, event, promotion, and reservations.

We keep `reservations` because it is known ahead of service and helps predict demand. We do not use `walk_ins` as a model feature because it is only known after customers arrive and would leak actual demand into the forecast.

In [71]:
import pandas as pd
import numpy as np

sales_df = pd.read_csv(DATA_DIR / 'historical_sales.csv', parse_dates=['timestamp'])
external_df = pd.read_csv(DATA_DIR / 'external_features.csv', parse_dates=['timestamp'])

feature_df = sales_df.merge(external_df, on='timestamp', how='left')
feature_df = feature_df.sort_values('timestamp').reset_index(drop=True)

feature_df['hour'] = feature_df['timestamp'].dt.hour
feature_df['day_of_week'] = feature_df['timestamp'].dt.dayofweek
feature_df['day_of_month'] = feature_df['timestamp'].dt.day
feature_df['month'] = feature_df['timestamp'].dt.month
feature_df['week_of_year'] = feature_df['timestamp'].dt.isocalendar().week.astype(int)
feature_df['is_weekend'] = (feature_df['day_of_week'] >= 5).astype(int)

feature_df['hour_sin'] = np.sin(2 * np.pi * feature_df['hour'] / 24)
feature_df['hour_cos'] = np.cos(2 * np.pi * feature_df['hour'] / 24)
feature_df['dow_sin'] = np.sin(2 * np.pi * feature_df['day_of_week'] / 7)
feature_df['dow_cos'] = np.cos(2 * np.pi * feature_df['day_of_week'] / 7)

feature_df['covers_lag_1'] = feature_df['covers'].shift(1)
feature_df['covers_lag_24'] = feature_df['covers'].shift(24)
feature_df['covers_lag_168'] = feature_df['covers'].shift(168)

feature_df['covers_roll_mean_3'] = feature_df['covers'].shift(1).rolling(window=3).mean()
feature_df['covers_roll_mean_24'] = feature_df['covers'].shift(1).rolling(window=24).mean()
feature_df['covers_roll_std_24'] = feature_df['covers'].shift(1).rolling(window=24).std()

feature_df['rain_flag'] = (feature_df['rain_mm'] > 0).astype(int)
feature_df['hot_flag'] = (feature_df['temp_c'] >= 30).astype(int)

model_feature_columns = [
    'timestamp', 'covers', 'reservations', 'promotion_flag',
    'holiday_flag', 'event_flag', 'temp_c', 'rain_mm', 'rain_flag', 'hot_flag',
    'hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'covers_lag_1', 'covers_lag_24', 'covers_lag_168',
    'covers_roll_mean_3', 'covers_roll_mean_24', 'covers_roll_std_24'
]

model_df_input = feature_df[model_feature_columns].dropna().reset_index(drop=True)

print('Raw merged dataset shape:', feature_df.shape)
print('Model-ready dataset shape after lag and rolling features:', model_df_input.shape)
model_df_input.head()


Raw merged dataset shape: (2160, 32)
Model-ready dataset shape after lag and rolling features: (1992, 26)


,timestamp,covers,reservations,promotion_flag,holiday_flag,event_flag,temp_c,rain_mm,rain_flag,hot_flag,...,hour_sin,hour_cos,dow_sin,dow_cos,covers_lag_1,covers_lag_24,covers_lag_168,covers_roll_mean_3,covers_roll_mean_24,covers_roll_std_24
0,2026-01-15 11:00:00,7,2,0,0,0,30.3,0.0,0,1,...,2.588190e-01,-0.965926,0.433884,-0.900969,36.0,8.0,15.0,26.333333,17.666667,10.825359
1,2026-01-15 12:00:00,12,5,0,0,0,21.2,0.0,0,0,...,1.224647e-16,-1.000000,0.433884,-0.900969,7.0,15.0,10.0,20.000000,17.625000,10.866032
2,2026-01-15 13:00:00,10,4,0,0,0,37.5,0.0,0,1,...,-2.588190e-01,-0.965926,0.433884,-0.900969,12.0,23.0,21.0,18.333333,17.500000,10.914689
3,2026-01-15 14:00:00,18,10,0,0,0,30.4,0.0,0,1,...,-5.000000e-01,-0.866025,0.433884,-0.900969,10.0,30.0,30.0,9.666667,16.958333,10.952384
4,2026-01-15 15:00:00,13,4,0,0,0,21.5,0.0,0,0,...,-7.071068e-01,-0.707107,0.433884,-0.900969,18.0,5.0,7.0,13.333333,16.458333,10.599340


## Step 4: Forecasting Model

We use a two-stage modeling approach. First, we build a simple baseline using the same hour from the previous week. This gives us a realistic benchmark that is easy to explain. Then we train an XGBoost regressor on the engineered features from Step 3.

XGBoost is a strong fit for this problem because restaurant demand forecasting is a tabular prediction task with nonlinear effects from time, weather, holidays, promotions, and recent demand history. It also retrains quickly, which makes it practical for a feedback-driven production loop.

For evaluation, we keep the split time-based. The last 7 days are used as a holdout period, which simulates forecasting an upcoming week without leaking future information into training.

In [ ]:
# EVAL : 1

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

feature_columns = [
    'reservations', 'promotion_flag',
    'holiday_flag', 'event_flag', 'temp_c', 'rain_mm', 'rain_flag', 'hot_flag',
    'hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'covers_lag_1', 'covers_lag_24', 'covers_lag_168',
    'covers_roll_mean_3', 'covers_roll_mean_24', 'covers_roll_std_24'
]

target_column = 'covers'
holdout_hours = 7 * 12

train_df = model_df_input.iloc[:-holdout_hours].copy()
test_df = model_df_input.iloc[-holdout_hours:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_test = test_df[feature_columns]
y_test = test_df[target_column]

baseline_pred = test_df['covers_lag_168'].copy()

xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = pd.Series(xgb_model.predict(X_test), index=test_df.index).clip(lower=0)

def evaluate_forecast(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    wape = np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()
    return {
        'model': model_name,
        'mae': round(float(mae), 3),
        'rmse': round(float(rmse), 3),
        'wape': round(float(wape), 3)
    }

metrics_df = pd.DataFrame([
    evaluate_forecast(y_test, baseline_pred, 'baseline_same_hour_last_week'),
    evaluate_forecast(y_test, xgb_pred, 'xgboost')
])

forecast_output = test_df[['timestamp', 'covers']].copy()
forecast_output = forecast_output.rename(columns={'covers': 'actual_covers'})
forecast_output['baseline_predicted_covers'] = baseline_pred.values
forecast_output['xgb_predicted_covers'] = xgb_pred.round(2).values
forecast_output['xgb_abs_error'] = (forecast_output['actual_covers'] - forecast_output['xgb_predicted_covers']).abs().round(2)

print('Train period:', train_df['timestamp'].min(), 'to', train_df['timestamp'].max())
print('Test period:', test_df['timestamp'].min(), 'to', test_df['timestamp'].max())
print('\nModel evaluation:')
display(metrics_df)

print('Sample predictions from the holdout period:')
display(forecast_output.head(12))


Train period: 2026-01-15 11:00:00 to 2026-06-22 22:00:00
Test period: 2026-06-23 11:00:00 to 2026-06-29 22:00:00

Model evaluation:


,model,mae,rmse,wape
0,baseline_same_hour_last_week,6.905,10.021,0.284
1,xgboost,3.261,4.524,0.134


Sample predictions from the holdout period:


,timestamp,actual_covers,baseline_predicted_covers,xgb_predicted_covers,xgb_abs_error
1908,2026-06-23 11:00:00,5,7.0,5.440000,0.44
1909,2026-06-23 12:00:00,17,15.0,20.480000,3.48
1910,2026-06-23 13:00:00,29,21.0,24.430000,4.57
1911,2026-06-23 14:00:00,17,25.0,20.940001,3.94
1912,2026-06-23 15:00:00,5,10.0,5.810000,0.81
1913,2026-06-23 16:00:00,19,31.0,13.300000,5.70
1914,2026-06-23 17:00:00,17,13.0,19.420000,2.42
1915,2026-06-23 18:00:00,14,9.0,17.260000,3.26
1916,2026-06-23 19:00:00,30,35.0,33.970001,3.97
1917,2026-06-23 20:00:00,9,13.0,11.020000,2.02


In [72]:
# EVAL 2
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

def add_advanced_features(df):
    df = df.copy()

    # Lag features
    df["covers_lag_1"] = df["covers"].shift(1)
    df["covers_lag_2"] = df["covers"].shift(2)
    df["covers_lag_24"] = df["covers"].shift(24)
    df["covers_lag_48"] = df["covers"].shift(48)
    df["covers_lag_168"] = df["covers"].shift(168)

    # Rolling stats
    df["covers_roll_mean_3"] = df["covers"].rolling(3).mean()
    df["covers_roll_mean_24"] = df["covers"].rolling(24).mean()
    df["covers_roll_std_24"] = df["covers"].rolling(24).std()

    # Momentum features (VERY IMPORTANT)
    df["momentum_1_24"] = df["covers_lag_1"] - df["covers_lag_24"]
    df["momentum_24_168"] = df["covers_lag_24"] - df["covers_lag_168"]

    # Reservation signal
    df["reservation_ratio"] = df["reservations"] / (df["covers_lag_24"] + 1)

    # Peak hour indicator
    df["is_peak_hour"] = df["hour"].isin([12,13,14,19,20,21]).astype(int)

    # Volatility
    df["rolling_max_24"] = df["covers"].rolling(24).max()
    df["rolling_min_24"] = df["covers"].rolling(24).min()

    # Weighted lag
    df["lag_weighted"] = (
        0.5 * df["covers_lag_1"] +
        0.3 * df["covers_lag_24"] +
        0.2 * df["covers_lag_168"]
    )

    df["hours_since_peak"] = np.maximum(0, df["hour"] - 20) 
    df["drop_flag"] = (df["covers_lag_1"] < df["covers_lag_2"]).astype(int)

    return df

model_df = add_advanced_features(model_df_input)

model_df = model_df.dropna().reset_index(drop=True)

In [73]:
feature_columns = [
    'reservations', 'promotion_flag',
    'holiday_flag', 'event_flag', 'temp_c', 'rain_mm', 'rain_flag', 'hot_flag',

    'hour', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',

    'covers_lag_1', 'covers_lag_2', 'covers_lag_24', 'covers_lag_48', 'covers_lag_168',
    'covers_roll_mean_3', 'covers_roll_mean_24', 'covers_roll_std_24',

    'momentum_1_24', 'momentum_24_168',
    'reservation_ratio', 'is_peak_hour',
    'rolling_max_24', 'rolling_min_24',
    'lag_weighted',"hours_since_peak", "drop_flag"
]

target_column = 'covers'

holdout_hours = 7 * 12

train_df = model_df.iloc[:-holdout_hours].copy()
test_df = model_df.iloc[-holdout_hours:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

In [74]:
y_train_log = np.log1p(y_train)
y_train_log

0       2.302585
1       3.044522
2       2.995732
3       3.367296
4       1.791759
          ...   
1735    2.197225
1736    2.708050
1737    3.258097
1738    3.178054
1739    2.079442
Name: covers, Length: 1740, dtype: float64

In [75]:
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [4, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9]
}

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    XGBRegressor(
        objective='reg:squarederror',
        eval_metric='mae',
        random_state=42
    ),
    param_distributions=param_grid,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    verbose=1,
    n_jobs=1
)

search.fit(X_train, y_train_log)

best_params = search.best_params_

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [76]:
from xgboost import XGBRegressor

best_model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='mae',
    early_stopping_rounds=30, 
    random_state=42,
    **best_params
)

In [77]:
split_idx = int(len(X_train) * 0.9)

X_tr, X_val = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
y_tr, y_val = y_train_log.iloc[:split_idx], y_train_log.iloc[split_idx:]

best_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",30
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [103]:
xgb_pred_log = best_model.predict(X_test)
xgb_pred = np.expm1(xgb_pred_log)

xgb_pred = pd.Series(xgb_pred, index=X_test.index).clip(lower=0)
xgb_pred = xgb_pred.clip(lower=0)
baseline_pred = test_df['covers_lag_168'].copy()

final_pred = np.where(
    xgb_pred > baseline_pred,
    xgb_pred,
    0.8 * xgb_pred + 0.2 * baseline_pred
)


def evaluate_forecast(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    wape = np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()

    return {
        "model": name,
        "mae": round(mae, 3),
        "rmse": round(rmse, 3),
        "wape": round(wape, 3)
    }


In [105]:
metrics_df = pd.DataFrame([
    evaluate_forecast(y_test, baseline_pred, "baseline"),
    evaluate_forecast(y_test, xgb_pred, "xgboost_optimized"),
    evaluate_forecast(y_test, final_pred, "ensemble")
])

print(metrics_df)

               model    mae    rmse   wape
0           baseline  6.905  10.021  0.284
1  xgboost_optimized  2.397   3.321  0.099
2           ensemble  2.550   3.441  0.105


In [95]:
forecast_output = test_df[['timestamp', 'covers']].copy()
forecast_output = forecast_output.rename(columns={'covers': 'actual_covers'})

forecast_output['baseline_pred'] = baseline_pred.values
forecast_output['xgb_pred'] = xgb_pred.values
forecast_output['final_pred'] = final_pred

forecast_output['abs_error'] = (
    forecast_output['actual_covers'] - forecast_output['final_pred']
).abs()

print(forecast_output.head(12))

               timestamp  actual_covers  baseline_pred   xgb_pred  final_pred  \
1740 2026-06-23 11:00:00              5            7.0   5.424937    5.739950   
1741 2026-06-23 12:00:00             17           15.0  19.838272   19.838272   
1742 2026-06-23 13:00:00             29           21.0  26.136190   26.136190   
1743 2026-06-23 14:00:00             17           25.0  21.098543   21.878836   
1744 2026-06-23 15:00:00              5           10.0   5.651213    6.520970   
1745 2026-06-23 16:00:00             19           31.0  14.488326   17.790661   
1746 2026-06-23 17:00:00             17           13.0  16.643438   16.643438   
1747 2026-06-23 18:00:00             14            9.0  16.404039   16.404039   
1748 2026-06-23 19:00:00             30           35.0  31.636202   32.308962   
1749 2026-06-23 20:00:00              9           13.0  10.403517   10.922814   
1750 2026-06-23 21:00:00             20           19.0  19.557297   19.557297   
1751 2026-06-23 22:00:00    

##### Staff prediction


In [ ]:
# staff planning considered with statistic relation like cover/x

import math
import pandas as pd
staff_roles_df = pd.read_csv(DATA_DIR / 'staff_roles.csv')
print(staff_roles_df.columns.tolist())
# Must have this structure
forecast_df = pd.DataFrame({
    "timestamp": test_df["timestamp"],
    "predicted_covers": xgb_pred
})

def calculate_staff_plan(forecast_df, staff_roles_df, buffer_factor=1.1):
    results = []

    for _, row in forecast_df.iterrows():
        covers = row["predicted_covers"]

        hour_data = {
            "timestamp": row["timestamp"],
            "predicted_covers": covers
        }

        total_cost = 0

        for _, role in staff_roles_df.iterrows():
            role_name = role["role"]
            station = role["station"]
            productivity = role["covers_per_hour_per_staff"]
            min_staff = role["min_staff_per_shift"]
            max_staff = role["max_staff_per_shift"]
            hourly_cost = role["hourly_cost"]

            # -------------------------
            # ROLE-SPECIFIC LOGIC
            # -------------------------

            if role_name == "server":
                load = covers

            elif role_name == "cook":
                # distribute kitchen load
                if station == "grill":
                    load = covers * 0.6
                elif station == "fryer":
                    load = covers * 0.4
                else:
                    load = covers

            elif role_name == "manager":
                # threshold-based staffing
                if covers < 30:
                    required = 1
                else:
                    required = 2

                cost = required * hourly_cost
                total_cost += cost

                hour_data[f"{role_name}_{station}"] = required
                hour_data[f"{role_name}_{station}_cost"] = cost
                continue

            else:
                load = covers

            # -------------------------
            # BASE CALCULATION
            # -------------------------

            required = math.ceil((load * buffer_factor) / productivity)

            # apply constraints
            required = max(required, min_staff)
            required = min(required, max_staff)

            # cost
            cost = required * hourly_cost
            total_cost += cost

            # store results
            col_name = f"{role_name}_{station}"
            hour_data[col_name] = required
            hour_data[f"{col_name}_cost"] = cost

        hour_data["total_hourly_cost"] = total_cost
        results.append(hour_data)

    return pd.DataFrame(results)


staff_plan_df = calculate_staff_plan(
    forecast_df,
    staff_roles_df,
    buffer_factor=1.1
)

staff_plan_df.head(12)

def smooth_staffing(df, role_columns, window=2):
    df = df.copy()
    for col in role_columns:
        df[col] = df[col].rolling(window, min_periods=1).max()
    return df

role_cols = [col for col in staff_plan_df.columns if "_" in col and "_cost" not in col]

staff_plan_df = smooth_staffing(staff_plan_df, role_cols)
print(staff_plan_df.head(12))

,timestamp,predicted_covers,server_dining,server_dining_cost,cook_grill,cook_grill_cost,cook_fryer,cook_fryer_cost,manager_floor,manager_floor_cost,total_hourly_cost
0,2026-06-23 11:00:00,5.424937,1,15.0,1,18.0,1,17.0,1,25.0,75.0
1,2026-06-23 12:00:00,19.838272,2,30.0,1,18.0,1,17.0,1,25.0,90.0
2,2026-06-23 13:00:00,26.136190,2,30.0,1,18.0,1,17.0,1,25.0,90.0
3,2026-06-23 14:00:00,21.098543,2,30.0,1,18.0,1,17.0,1,25.0,90.0
4,2026-06-23 15:00:00,5.651213,1,15.0,1,18.0,1,17.0,1,25.0,75.0
5,2026-06-23 16:00:00,14.488326,1,15.0,1,18.0,1,17.0,1,25.0,75.0
6,2026-06-23 17:00:00,16.643438,2,30.0,1,18.0,1,17.0,1,25.0,90.0
7,2026-06-23 18:00:00,16.404039,2,30.0,1,18.0,1,17.0,1,25.0,90.0
8,2026-06-23 19:00:00,31.636202,2,30.0,1,18.0,1,17.0,2,50.0,115.0
9,2026-06-23 20:00:00,10.403517,1,15.0,1,18.0,1,17.0,1,25.0,75.0
